#### Реальный кейс: Мониторинг температуры и вибрации подшипника

Напишем код, который симулирует работу датчиков станка и находит момент,  
когда подшипник начинает разрушаться. Использовать будем самый эффективный   
алгоритм — Isolation Forest.

Шаг 1: Симулируем нормальную работу и зарождение поломки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

# Генерируем 1000 временных меток (например, показания раз в 1 час)
np.random.seed(42)
n_readings = 1000

# Симулируем нормальную работу станка (небольшие случайные колебания)
vibration = np.random.normal(loc=2.0, scale=0.2, size=n_readings)     # Вибрация вокруг 2.0 мм/с
temperature = np.random.normal(loc=65.0, scale=1.5, size=n_readings)  # Температура вокруг 65°C

# Симулируем аномалию (последние 40 часов подшипник начинает греться и вибрировать перед поломкой)
vibration[-40:] = vibration[-40:] + np.random.uniform(1.0, 2.5, size=40)
temperature[-40:] = temperature[-40:] + np.random.uniform(10.0, 25.0, size=40)

df_machinery = pd.DataFrame({'Vibration': vibration, 'Temperature': temperature})

df_machinery.head()


Шаг 2: Обучаем IsolationForest

Главный параметр здесь — contamination (уровень загрязнения). Это процент аномалий,  
который мы ожидаем увидеть в данных. На производстве это обычно доли процента   
(например, 0.01), но для нашего теста поставим 0.05 (5%).

In [ ]:
# Инициализируем модель
# В отличие от обучения с учителем, мы не делим данные на X_train и y_train!
model_if = IsolationForest(contamination=0.05, random_state=42)

# Обучаем модель на всем массиве данных и сразу получаем предсказания
# Модель вернет: 1 для нормы, -1 для аномалии
df_machinery['Anomaly_Score'] = model_if.fit_predict(df_machinery[['Vibration', 'Temperature']])

# Посмотрим, сколько аномалий нашел алгоритм
print(df_machinery['Anomaly_Score'].value_counts())

Шаг 3: Визуализация — ловим поломку на графике

Построим график изменения температуры и подсветим красными точками места, где алгоритм Isolation Forest заподозрил неладное.

In [ ]:
plt.figure(figsize=(12, 5))

# Строим линию температуры
plt.plot(df_machinery.index, df_machinery['Temperature'], color='blue', label='Температура (°C)', alpha=0.6)

# Накладываем красные точки поверх линии в местах аномалий
anomalies = df_machinery[df_machinery['Anomaly_Score'] == -1]
plt.scatter(anomalies.index, anomalies['Temperature'], color='red', label='Зафиксированная аномалия', s=40)

plt.xlabel('Время (часы)')
plt.ylabel('Показания датчика')
plt.title('Мониторинг состояния оборудования и обнаружение аномалий')
plt.legend()
plt.grid(True)
plt.show()

Как алгоритм отличает случайный единичный «чих» оборудования (например, упавший гаечный ключ) от реальной зарождающейся поломки?

Шаг 4: Симулируем случайный «шум» (удар ключа)

Возьмем наш код из предыдущего шага и добавим в него одиночный мощный выброс в самом начале, когда оборудование работает абсолютно исправно.

In [ ]:
df_noisy = df_machinery.copy()

# На 200-м часу произошел случайный единичный удар (вибрация взлетела но сразу вернулась в норму)
df_noisy.loc[200, 'Vibration'] = 6.5    # crit
df_noisy.loc[200, 'Temperature'] = 66.0 # normal

Если мы сейчас просто прогоним через это IsolationForest, он гарантированно отметит 200-й час как аномалию (-1).  
С точки зрения геометрии, эта точка висит далеко от общего облака, и алгоритм прав. Но для завода это ложная тревога.

Шаг 5: Магическое решение — Скользящее окно (Moving Window)

Чтобы отсечь случайные одиночные всплески, инженеры данных никогда не отправляют в модель «сырые» секундные или   
часовые показания. Данные сначала сглаживают с помощью скользящего среднего или оценивают количество аномалий за   
интервал времени. Мы применим стратегию «X аномалий за последние Y часов».

In [ ]:
# 1. Переобучаем модель на зашумленных данных
model_if_noisy = IsolationForest(contamination=0.05, random_state=42)
df_noisy['Raw_Anomaly'] = model_if_noisy.fit_predict(df_noisy[['Vibration', 'Temperature']])

# Переводим метки в более удобный формат: 1 — аномалия, 0 — норма
df_noisy['Is_Anomaly'] = (df_noisy['Raw_Anomaly'] == -1).astype(int)

# 2. Применяем скользящее окно (например, за последние 5 часов)
# Считаем, сколько раз алгоритм заподозрил аномалию за этот промежуток
window_size = 5
df_noisy['Anomaly_Density'] = df_noisy['Is_Anomaly'].rolling(window=window_size).sum()

# 3. Финальное бизнес-правило: включаем сирену, только если аномалии идут кучно
# Например, если из последних 5 часов система ругалась минимум 3 раза
df_noisy['FINAL_ALARM'] = df_noisy['Anomaly_Density'] >= 3

# Проверяем точку случайного удара
print("Данные на 200-м часу (удар ключа):")
print(df_noisy.loc[200, ['Vibration', 'Raw_Anomaly', 'FINAL_ALARM']])

# Проверяем точку реальной поломки в конце
print("\nДанные на 980-м часу (разрушение подшипника):")
print(df_noisy.loc[980, ['Vibration', 'Raw_Anomaly', 'FINAL_ALARM']])

## Решение через нейросеть: Топ-3 архитектуры для этой задачи

1. **Autoencoder (Автоэквилибриум / Автокодировщик)** — _Идеально для старта_.  
    - **Как работает:** Сеть сжимает данные от датчиков в маленькую "шпаргалку", а затем пытается восстановить    
    их обратно. Мы обучаем её **только на здоровых данных**.  
    - **Поиск аномалий:** Когда на вход поступает поломка, сеть не может её правильно восстановить. Разница между   
     входом и выходом (ошибка восстановления) резко растет — это и есть аномалия.  

2. **LSTM Autoencoder (или GRU)**  
    - **Как работает:** То же самое, что и обычный автокодировщик, но внутри используются рекуррентные слои (LSTM).  
    - **Зачем нужен:** Если ваши аномалии зависят от времени (например, плавающий тренд температуры в течение 5 часов,  
     а не резкий скачок).  
3. **1D-CNN (Одномерные сверточные сети)**
    - **Как работает:** Сеть сканирует сигналы датчиков (как картинку) в поисках аномальных паттернов или частот   
    (например, специфический стук или вибрация подшипника).

### Реализация Автоэнкодера на Python (Keras/TensorFlow)

Шаг 1: Подготовка данных

Для нейросетей масштабирование критично на 100%. Приведем данные к диапазону от 0 до 1. И,   
самое главное, обучать модель мы будем только на первых 800 часах (где станок гарантированно   
здоров).

Для **нейронных сетей** правила масштабирования данных принципиально отличаются от классического 
машинного обучения (типа SVM или линейных моделей).

Здесь выбор скейлера зависит не просто от формы данных, а от **функций активации** на скрытых и   
выходных слоях, а также от проблемы **затухания или взрыва градиентов**.

Вот лучшие практики (Best Practices) и фавориты именно для глубокого обучения (Deep Learning).

---

🔥 Главные фавориты для нейросетей

1. `StandardScaler` — Король для скрытых слоев с ReLU / GELU (По умолчанию)

Если ваша сеть состоит из скрытых слоев с активациями `ReLU`, `LeakyReLU` или `GELU` (как в   
большинстве современных полносвязных сетей, CNN и трансформеров), вашим выбором по умолчанию  
 должен быть **`StandardScaler`**.

- **Почему**: Он центрирует данные вокруг `0`. Нейросети гораздо быстрее обучаются и стабильнее  
 обновляют веса, когда входы имеют нулевое среднее (это снижает эффект "зигзагообразного" обновления   
 весов при градиентном спуске).
 
- **Диапазон**: Данные не ограничены жестко, но ~99% значений будут лежать в пределах от `-3` до `+3`.   
Это идеально для стабильной работы весов.

2. `MinMaxScaler` — Выбор для Sigmoid / Tanh и Автокодировщиков

Его используют тогда, когда архитектура или физика процесса требуют жестких границ.

- **С активацией Sigmoid**: На входе или на выходе. Сигмоида «живет» в диапазоне `(0, 1)`. Если подать на   
вход `StandardScaler` со значением `+10`, сигмоида намертво уйдет в насыщение (градиент станет равен 0,   
нейрон «умрет»). `MinMaxScaler` держит данные в чувствительной зоне.

- **С активацией Tanh**: В этом случае `MinMaxScaler` настраивают на диапазон от `-1` до `1`:   
`MinMaxScaler(feature_range=(-1, 1))`. Это стандарт для **LSTM** и **RNN** сетей при работе со временем  и текстом.

- **Генеративные сети и Автокодировщики (Autoencoders)**: Если вы восстанавливаете картинку (пиксели 0–255) или те  
 же сигналы станков, на выходном слое часто ставят `Sigmoid`. Соответственно, и таргет, и вход сжимают через `MinMaxScaler`  
  в диапазон `[0, 1]`.

---

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.preprocessing import MinMaxScaler

# 1. Масштабируем признаки (Выбрация и Температура)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_machinery[['Vibration', 'Temperature']])

# 2. Разделяем: берем первые 800 часов чистой работы ДЛЯ ОБУЧЕНИЯ
X_train = scaled_data[:800]

# Тестировать модель будем НА ВСЕХ данных (все 1000 часов, включая поломку в конце)
X_test = scaled_data

Шаг 2: Создание архитектуры Автоэнкодера

In [ ]:
# Размерность входных данных (у нас 2 датчика)
input_dim = X_train.shape[1]

# Входной слой
input_layer = Input(shape=(input_dim,))

# ЭНКОДЕР: сжимаем 2 признака в 1 промежуточное число
# Магический метод __call__ 
encoded = Dense(1, activation='relu')(input_layer)